In [2]:
# Parameters
name = "t"
Email = "t"
Organization = "SMVDU"
EmployeeID = "y"


In [3]:
import sqlite3
import numpy as np
import os
import cv2
import numpy as np

DB_PATH = 'data/employees.db'
os.makedirs('data', exist_ok=True)

def get_connection():
    return sqlite3.connect(DB_PATH)
def init_db():
    """Create tables if they don't exist."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS employees (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_id TEXT    NOT NULL,
            name        TEXT    NOT NULL,
            email       TEXT    NOT NULL,
            organization TEXT   NOT NULL,
            face_vector BLOB    NOT NULL
        )
    ''')
    conn.commit()
    conn.close()
    print("Database initialised at:", DB_PATH)
init_db()

video = cv2.VideoCapture(0)
facedetect = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')

faces_data = []
i = 0
while True:
    ret, frame = video.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = facedetect.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        crop_img   = frame[y:y+h, x:x+w, :]
        resized_img = cv2.resize(crop_img, (50, 50))
        if len(faces_data) <= 50 and i % 10 == 0:
            faces_data.append(resized_img)
        i += 1
        cv2.putText(frame, str(len(faces_data)), (50, 50),cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255), 1)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 1)
    imgBackground = cv2.imread("background.png")
    imgBackground[162:162+480, 55:55+640] = frame
    cv2.imshow("Frame", imgBackground)
    k = cv2.waitKey(1)
    if k == ord('q') or len(faces_data) == 50:
        break
video.release()
cv2.destroyAllWindows()

faces_data = np.asarray(faces_data)
faces_data = faces_data.reshape(50, -1)   


if faces_data.shape == (50, 7500):
    conn   = get_connection()
    cursor = conn.cursor()
    rows = [
        ( EmployeeID,name, Email,Organization,
            faces_data[j].tobytes()   
        )
        for j in range(50)
    ]

    cursor.executemany(
        '''
        INSERT INTO employees (employee_id, name, email, organization, face_vector)
        VALUES (?, ?, ?, ?, ?)
        ''',
        rows
    )
    conn.commit()

    cursor.execute('SELECT COUNT(*) FROM employees WHERE employee_id = ?', (EmployeeID,))
    count = cursor.fetchone()[0]
    conn.close()
    print(f"Saved {count} face vectors for '{name}' (ID: {EmployeeID}) to SQLite.")
else:
    print("Feature vector shape mismatch; cannot insert into DB!")


Database initialised at: data/employees.db
Saved 50 face vectors for 't' (ID: y) to SQLite.


In [3]:
import pandas as pd
conn = get_connection()
df = pd.read_sql_query(
    'SELECT id, employee_id, name, email, organization FROM employees',
    conn
)
conn.close()

print(f"Total records in DB: {len(df)}")
print(df.groupby(['employee_id', 'name', 'email', 'organization']).size().reset_index(name='face_samples'))

Total records in DB: 50
  employee_id    name                    email organization  face_samples
0    23bcs061  Piyush  infopiush2004@gmail.com       GOOGLE            50


In [4]:
def load_faces_from_db():
    """
    Returns:
        faces        – np.ndarray of shape (N, 7500)
        names        – list of names (length N)
        employee_ids – list of IDs (length N)
    """
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT name, employee_id, face_vector FROM employees')
    rows   = cursor.fetchall()
    conn.close()

    names        = [r[0] for r in rows]
    employee_ids = [r[1] for r in rows]
    faces        = np.array([
        np.frombuffer(r[2], dtype=np.uint8) for r in rows
    ])  # shape: (N, 7500)

    return faces, names, employee_ids

faces, names, ids = load_faces_from_db()
print("Loaded faces shape:", faces.shape)
print("Sample names:", names[:3], "...", names[-3:])
print(faces)


Loaded faces shape: (50, 7500)
Sample names: ['Piyush', 'Piyush', 'Piyush'] ... ['Piyush', 'Piyush', 'Piyush']
[[ 28  32  32 ...  79  85  88]
 [ 42  43  39 ...  83  87  88]
 [ 46  50  45 ...  87  90  96]
 ...
 [145 146 140 ...  29  25  30]
 [148 145 140 ...  20  23  24]
 [126 127 121 ...  18  22  27]]
